# 01 필터 감사 — 본분석 코퍼스 확정 (LDA·의제비중·감성 공용 우주)

`분석토큰_언론사_{기간}.csv`(토큰)와 `전처리_본문_언론사_{기간}.csv`(플래그·길이)를 `article_id`로 붙여,
02~05 전 단계가 **공유할 본분석 우주(universe)** 를 한 번에 확정하는 노트북

- 우주 = 5단계 필터 통과분 — ① weather/closing → ② within_dup → ③ cross_dup 제외 → ④ `n_tokens<10` 제거 → ⑤ 소프트뉴스(`스포츠`·`연예`) 제외
- 행을 물리적으로 버리지 않고 `in_universe` 플래그로 표시 — cross_dup 포함 민감도·소프트 포함 민감도를 같은 파일에서 재구성 가능
- 산출 — `result/분석코퍼스_*.csv`(전 행+플래그), `result/필터절단율_*.csv`(단계별 표), `result/미분류_수기코딩_*.csv`(LDA 독립 hard/soft 코딩 대상)


## 분모 라벨 (혼동 방지 — 표·문장마다 어느 기준인지 명시)

| 라벨 | 정의 | 기대값 |
|---|---|---|
| `raw` | 전체 | 14,057 |
| `raw−소프트만` | 소프트뉴스만 제외 | 13,231 |
| `dedup+wxc+소프트` | n_tokens 컷 전(①②③⑤) | 10,638 |
| `최종 우주` | 5단계 전부(①②③④⑤) | 10,616 |

플래그 출처가 단계마다 다름 — dedup·weather/closing은 **전처리본 플래그**, 소프트뉴스는 **`article_category` 기반**


In [ ]:
# Colab에서 실행할 때만 아래 3줄 주석 해제 — 로컬/WSL에서는 그대로 두기
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, unicodedata
import pandas as pd

# Colab 기본 경로 — Drive 미마운트면 except로 빠져 로컬/WSL 경로 사용
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR)
DATA_DIR = PROJECT_DIR / 'news'
RESULT_DIR = PROJECT_DIR / 'result'
RESULT_DIR.mkdir(exist_ok=True)

def normalize_name(p): return unicodedata.normalize('NFC', p.name)
def find_one(pat):
    cands = sorted(p for p in DATA_DIR.iterdir() if p.is_file() and re.match(pat, normalize_name(p)))
    if not cands:
        raise FileNotFoundError(f'{pat} 매칭 파일 없음: {DATA_DIR}')
    return cands[-1]

TOKEN_PATH = find_one(r'^분석토큰_언론사_\d{6}_\d{6}\.csv$')
PRE_PATH   = find_one(r'^전처리_본문_언론사_\d{6}_\d{6}\.csv$')
PERIOD = re.search(r'(\d{6}_\d{6})', normalize_name(TOKEN_PATH)).group(1)
print('토큰:', TOKEN_PATH.name)
print('전처리본:', PRE_PATH.name, '/ 기간:', PERIOD)


In [ ]:
# --- 두 CSV 로드 (★ utf-8-sig — 토큰 CSV 헤더 BOM이라 utf-8로 읽으면 article_id KeyError) ---
# 토큰파일 = 토큰·메타, 전처리본 = 플래그·길이(거대한 body/body_cleaned/text는 안 읽음)
# pandas로 읽어 csv.field_size_limit 불필요 — 표준 csv로 전처리본 읽을 땐 limit 상향 필요
tok = pd.read_csv(TOKEN_PATH, encoding='utf-8-sig')

FLAG_COLS = ['is_weather','is_closing','is_within_press_dup','is_cross_press_dup']
PRE_USECOLS = ['article_id','article_category_full','body_length'] + FLAG_COLS
pre = pd.read_csv(PRE_PATH, encoding='utf-8-sig', usecols=PRE_USECOLS)

print('토큰 행수:', len(tok), '/ 컬럼:', list(tok.columns))
print('전처리본 행수:', len(pre))

# ★ 플래그는 1/0 아니라 문자열 'True'/'False' — 읽자마자 bool 정규화(일반 truthiness면 'False'도 True 취급돼 우주 N 틀어짐)
BOOL_MAP = {'True':True,'False':False,True:True,False:False}
for c in FLAG_COLS:
    pre[c] = pre[c].map(BOOL_MAP)
    bad = pre[c].isna().sum()
    assert bad == 0, f'{c} 에 True/False 아닌 값 {bad}개 — 매핑 실패'
print('플래그 bool 정규화 완료 — 예외값 0')


In [ ]:
# --- article_id 1:1 join 검증 후 병합 ---
assert tok['article_id'].is_unique, '토큰 article_id 중복'
assert pre['article_id'].is_unique, '전처리본 article_id 중복'
assert set(tok['article_id']) == set(pre['article_id']), '두 파일 article_id 집합 불일치'

df = tok.merge(pre, on='article_id', how='inner', validate='one_to_one')
assert len(df) == len(tok) == len(pre), f'join 후 행수 변동 {len(df)}'
print('join 완료 — 행수:', len(df))

GROUPS = ['경제','통신·보도','정치색','지상파']  # ★ 통신·보도는 U+00B7 단일 그룹명 — 쪼개거나 일반 점과 혼동 금지
assert set(df['media_group'].unique()) == set(GROUPS), f"그룹 불일치: {sorted(df['media_group'].unique())}"


In [ ]:
# --- 파생 플래그 + 5단계 필터 ---
SOFT_CATS = {'스포츠','연예'}   # ★ 실데이터 값은 '연예'(='연예인' 아님). literal 틀리면 365건 안 빠짐
df['is_soft']    = df['article_category'].isin(SOFT_CATS)
df['is_미분류']  = df['article_category'].eq('미분류')
df['n_tokens']   = df['n_tokens'].astype(int)

keep_wxc   = ~(df['is_weather'] | df['is_closing'])           # ① weather/closing 제거
keep_within = keep_wxc & ~df['is_within_press_dup']           # ② within_dup 제거
keep_cross  = keep_within & ~df['is_cross_press_dup']         # ③ cross_dup 제거(=본분석)
keep_short  = keep_cross & (df['n_tokens'] >= 10)             # ④ 짧은 문서 제거
keep_soft   = keep_short & ~df['is_soft']                     # ⑤ 소프트뉴스 제거
df['in_universe'] = keep_soft                                 # 최종 우주(5단계 전부)

# 단계별 잔존 그룹 수 표 — 단계가 어느 그룹을 깎는지(우주 N 재산출, 하드코딩 아님)
stage_masks = {
    'joined': pd.Series(True, index=df.index),
    '①weather/closing': keep_wxc,
    '②within_dup': keep_within,
    '③cross_dup': keep_cross,
    '④n_tokens>=10': keep_short,
    '⑤소프트(최종 우주)': keep_soft,
}
audit = pd.DataFrame({name: df[m].groupby(df['media_group']).size().reindex(GROUPS, fill_value=0)
                      for name, m in stage_masks.items()}).T
audit['합계'] = audit.sum(axis=1)
print(audit)


In [ ]:
# --- 우주 N 기대치 대조 + 분모 라벨 4개 + 절단율 보고 ---
EXPECTED = {'경제':2362,'정치색':1996,'지상파':2425,'통신·보도':3833}  # codex/서브에이전트 검증 기대치
univ = df[df['in_universe']].groupby('media_group').size().reindex(GROUPS).to_dict()
print('최종 우주 N:', univ, '/ 합계', sum(univ.values()))
ok = all(univ[g] == EXPECTED[g] for g in GROUPS)
print('기대치 대조:', 'PASS' if ok else f'MISMATCH (기대 {EXPECTED})')

# 분모 라벨 4개 — 표·문장에서 어느 기준인지 못박기
raw_n          = len(df)
raw_minus_soft = int((~df['is_soft']).sum())
dedup_wxc_soft = int((keep_cross & ~df['is_soft']).sum())   # n_tokens 컷 전(①②③⑤)
final_univ     = int(df['in_universe'].sum())
print(f'\n[분모 라벨] raw={raw_n} / raw−소프트만={raw_minus_soft} / dedup+wxc+소프트(n컷 전)={dedup_wxc_soft} / 최종 우주={final_univ}')

# 단계별 그룹 절단 수(전 단계 대비 제거량) — dedup이 어느 그룹을 크게 깎는지
cut = audit.drop(columns='합계').diff().mul(-1).iloc[1:].astype(int)
print('\n[단계별 그룹 절단 수]')
print(cut)


In [ ]:
# --- 소프트뉴스 절단율 비대칭 + 통신·보도 0건 점검 ---
pre_soft_n = df[keep_short].groupby('media_group').size().reindex(GROUPS)      # 소프트 제거 직전 N
soft_removed = df[keep_short & df['is_soft']].groupby('media_group').size().reindex(GROUPS, fill_value=0)
soft_rate = (soft_removed / pre_soft_n * 100).round(1)
print('[소프트뉴스 절단율 (소프트 직전 N 대비, %)]')
print(pd.DataFrame({'직전N':pre_soft_n,'소프트제거':soft_removed,'절단율%':soft_rate}))

# ★ 통신·보도 0%는 '연성을 안 써서'가 아니라 카테고리 스킴 아티팩트 — raw category에 스포츠·연예 섹션 부재 점검
raw_soft_tongsin = df[(df['media_group']=='통신·보도') & df['is_soft']].shape[0]
print('\n통신·보도 raw 소프트(스포츠/연예) 건수:', raw_soft_tongsin, '(0이면 카테고리 스킴 아티팩트 — 본문 한계 명시)')
social_life = df[df['in_universe'] & df['article_category'].isin(['사회','생활'])].groupby('media_group').size().reindex(GROUPS)
univ_s = pd.Series(univ).reindex(GROUPS)
print('우주 내 사회+생활 비중(%) — 통신·보도 연성 흡수 점검:')
print((social_life / univ_s * 100).round(1))


In [ ]:
# --- 산출 저장 ---
# 1) 본분석 코퍼스 — 전 행 + 토큰 + 플래그(거대한 body/text 제외). 다운스트림은 in_universe로 필터, 민감도는 플래그로 재구성
OUT_COLS = ['article_id','media_group','press','date','article_category','article_category_full',
            'title_cleaned','n_tokens','tokens','body_length',
            'is_weather','is_closing','is_within_press_dup','is_cross_press_dup','is_soft','is_미분류','in_universe']
corpus_path = RESULT_DIR / f'분석코퍼스_언론사_{PERIOD}.csv'
df[OUT_COLS].to_csv(corpus_path, index=False, encoding='utf-8-sig')
print('저장:', corpus_path.name, '/', len(df), '행 (in_universe True:', int(df['in_universe'].sum()), ')')

# 2) 필터 단계별 표 — 잔존 수(survivors)와 절단 수(removed) 둘 다 저장(파일명-내용 일치)
surv_path = RESULT_DIR / f'필터단계별잔존_언론사_{PERIOD}.csv'
audit.to_csv(surv_path, encoding='utf-8-sig')
cut_path = RESULT_DIR / f'필터절단수_언론사_{PERIOD}.csv'
cut.to_csv(cut_path, encoding='utf-8-sig')
print('저장:', surv_path.name, '(잔존) /', cut_path.name, '(절단)')


In [ ]:
# 3) 미분류 수기코딩 대상 — ★ dominant 사후할당(내생성) 말고 제목/원문으로 LDA 독립 판정
# 우주에 든 미분류만, body_cleaned 발췌 포함해서 내보냄(코더가 hard/soft 채움)
uncls = df[df['in_universe'] & df['is_미분류']][['article_id','media_group','press','article_category','title_cleaned']].copy()
print('우주 내 미분류:', len(uncls), '건  / 그룹 분포:', uncls['media_group'].value_counts().to_dict())

# body_cleaned 발췌만 따로 읽어 붙임(전처리본에서 해당 id만)
body = pd.read_csv(PRE_PATH, encoding='utf-8-sig', usecols=['article_id','body_cleaned'])
uncls = uncls.merge(body, on='article_id', how='left')
uncls['body_excerpt'] = uncls['body_cleaned'].astype(str).str.slice(0, 200)
uncls = uncls.drop(columns='body_cleaned')
uncls['manual_hardsoft'] = ''   # 코더가 'hard'/'soft' 채움 — 판정 규칙·블라인드는 별도 기록
uncls_path = RESULT_DIR / f'미분류_수기코딩_언론사_{PERIOD}.csv'
uncls.to_csv(uncls_path, index=False, encoding='utf-8-sig')
print('저장:', uncls_path.name, '(manual_hardsoft 채워 재반영 — soft 판명분은 소프트 절단에 합산)')


## 검증 체크리스트
- join 1:1 — 두 파일 article_id 집합 일치·중복 없음 (셀에서 assert)
- 플래그 bool 정규화 — `'True'/'False'`→bool, 예외값 0 (assert)
- 우주 N 기대치 대조 — `경제2362·정치색1996·지상파2425·통신·보도3833`, 합계 10,616 → **PASS** 떠야 함
- 분모 라벨 4개(14057 / 13231 / 10638 / 10616)가 표에 라벨과 함께 찍히는지
- 단계별 절단 — dedup이 통신·보도·지상파를 크게 깎고 정치색은 거의 안 깎임
- 통신·보도 raw 소프트 0건(카테고리 아티팩트) + 사회·생활 비중 비대칭 → 본문 한계로 명시
- 미분류 수기코딩 대상은 LDA 결과와 무관하게 제목/원문으로만 판정
